# NLP
## Exercise 2: Embeddings

Wojtek Kutak

In [1]:
from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import re
from sentence_transformers import SentenceTransformer
import gensim.downloader as api
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

phrase_regex = re.compile(r'\b\w+\b')
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

d:\Dev\nlp-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("stanfordnlp/imdb")
del ds['unsupervised']

### Dataset prepocessing

I am going to test 2 techniques:
- lemmatization - reducing a word to its base or dictionary form, called a lemma (changing, change, changed -> change)
- stemming - keeping only the core (changing, change, changed -> chang)

In [3]:
def stem_text(texts: list[str]) -> list[str]:
    texts = [phrase_regex.findall(text.lower()) for text in texts]
    texts = [[stemmer.stem(word) for word in text] for text in texts]
    return texts

def lemmatize_text(texts: list[str]) -> list[str]:
    texts = [phrase_regex.findall(text.lower()) for text in texts]
    texts = [[lemmatizer.lemmatize(word) for word in text] for text in texts]
    return texts

In [4]:
# ds_mini = DatasetDict()
# ds_mini['train'] = ds.shuffle(seed=42)['train'].select(range(10))
# ds_mini['test'] = ds.shuffle(seed=42)['test'].select(range(10))
# ds_mini

In [5]:
ds_stemmed = DatasetDict({
    'train': {
        'text': stem_text(ds['train']['text']),
        'label': ds['train']['label']
    },
    'test': {
        'text': stem_text(ds['test']['text']),
        'label': ds['test']['label']
    }
})

ds_lemmatized = DatasetDict({
    'train': {
        'text': lemmatize_text(ds['train']['text']),
        'label': ds['train']['label']
    },
    'test': {
        'text': lemmatize_text(ds['test']['text']),
        'label': ds['test']['label']
    }
})

In [6]:
datasets = {
    "Stemmed": ds_stemmed,
    "Lemmatized": ds_lemmatized
}

## Embeddings
- word2vec
- glove
- fasttext

In [7]:
################################## Word2Vec ##################################
def get_word2vec():
    print("Word2Vec model loaded")
    return api.load("word2vec-google-news-300")

def process_word2vec_model(model, text: list[list[str]]):
    words = [np.array([model[word] for word in sublist if word in model]) for sublist in text]
    return words

################################## GloVe ##################################
def get_glove():
    print("GloVe model loaded")
    return api.load("glove-wiki-gigaword-300")

def process_glove_model(model, text: list[list[str]]):
    words = [np.array([model[word] for word in sublist if word in model]) for sublist in text]
    return words

################################## FASTTEXT ##################################
def get_fasttext():
    print("FastText model loaded")
    return api.load("fasttext-wiki-news-subwords-300")

def process_fasttext_model(model, text: list[list[str]]):
    words = [np.array([model[word] for word in sublist if word in model]) for sublist in text]
    return words


## Models:
- RandomForest
- Logistic regression
- XGBoost
- Support Vector Machine

In [8]:
datasets = {
    "Stemmed": ds_stemmed,
    "Lemmatized": ds_lemmatized
}

embedding_models = {
    "Word2Vec": (get_word2vec, process_word2vec_model),
    "GloVe": (get_glove, process_glove_model)
}

models = {
    "RandomForest": RandomForestClassifier(),
    "Logistic regression": LogisticRegression(),
    "XGBoost": XGBClassifier(),
    "SVC": SVC()
}
results = {"Preprocessing": [], "Embedding Model": [], "Classifier": [], "Accuracy": [],
                            "Precision": [], "Recall": [], "F1-Score": []}

In [9]:
def evaluate_models(results: dict[str, list], datasets: dict[str, DatasetDict]):
    for emb_name, (get_emb_func, process_model_func) in embedding_models.items():
        emb_model = get_emb_func()
        for ds_name, ds in datasets.items():
            X_train_emb_list = process_model_func(emb_model, ds['train']['text'])
            X_train_emb = np.array([x.mean(axis=0) for x in X_train_emb_list])
            y_train = np.array(ds['train']['label'])
            X_test_emb_list = process_model_func(emb_model, ds['test']['text'])
            X_test_emb = np.array([x.mean(axis=0) for x in X_test_emb_list])
            y_test = np.array(ds['test']['label'])

            for model_name, model in models.items():
                print(f"Processing dataset: '{ds_name}' with embedding model: '{emb_name}' "
                    f"and classifier: '{model_name}'")
                model.fit(X_train_emb, y_train)
                y_pred = model.predict(X_test_emb)
                precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
                accuracy = accuracy_score(y_test, y_pred)
                model_metrics = {
                    "Preprocessing": ds_name,
                    "Embedding Model": emb_name,
                    "Classifier": model_name,
                    "Accuracy": accuracy,
                    "Precision": precision,
                    "Recall": recall,
                    "F1-Score": f1
                }
                for key in model_metrics:
                    results[key].append(model_metrics[key])
evaluate_models(results, datasets)

Word2Vec model loaded
Processing dataset: 'Stemmed' with embedding model: 'Word2Vec' and classifier: 'RandomForest'
Processing dataset: 'Stemmed' with embedding model: 'Word2Vec' and classifier: 'Logistic regression'
Processing dataset: 'Stemmed' with embedding model: 'Word2Vec' and classifier: 'XGBoost'
Processing dataset: 'Stemmed' with embedding model: 'Word2Vec' and classifier: 'SVC'
Processing dataset: 'Lemmatized' with embedding model: 'Word2Vec' and classifier: 'RandomForest'
Processing dataset: 'Lemmatized' with embedding model: 'Word2Vec' and classifier: 'Logistic regression'
Processing dataset: 'Lemmatized' with embedding model: 'Word2Vec' and classifier: 'XGBoost'
Processing dataset: 'Lemmatized' with embedding model: 'Word2Vec' and classifier: 'SVC'
GloVe model loaded
Processing dataset: 'Stemmed' with embedding model: 'GloVe' and classifier: 'RandomForest'
Processing dataset: 'Stemmed' with embedding model: 'GloVe' and classifier: 'Logistic regression'
Processing dataset: 

# Contextual embeddings

In order to improve performance, I am going to test the impact of the contextual embeddings, 
Sentence-BERT specifically.

In [16]:
################################## SENTENCE-BERT ##################################
def get_sbert():
    print("Sentence-BERT model loaded")
    return SentenceTransformer('all-MiniLM-L6-v2')

def process_sbert(model: SentenceTransformer, texts: list[str]):
    sentences = [text.split('.')[:-1] for text in texts]
    sentences = [[s for s in sentence if s] for sentence in sentences]  # Remove empty lists
    embedding = [model.encode(sentence) for sentence in sentences]
    return embedding

In [22]:
def evaluate_sbert(results: dict[str, list], ds: DatasetDict):
    sbert = get_sbert()
    X_train_emb_list = process_sbert(sbert, ds['train']['text'])
    train_nonzero_idx = [idx for idx, x in enumerate(X_train_emb_list) if x.size > 0]
    X_train_emb = np.array([X_train_emb_list[i].mean(axis=0) for i in train_nonzero_idx])
    y_train = np.array(ds['train']['label'][train_nonzero_idx])
    print(f"{X_train_emb.shape=} {y_train.shape=}")
    X_test_emb_list = process_sbert(sbert, ds['test']['text'])
    test_nonzero_idx = [idx for idx, x in enumerate(X_test_emb_list) if x.size > 0]
    X_test_emb = np.array([X_test_emb_list[i].mean(axis=0) for i in test_nonzero_idx])
    y_test = np.array(ds['test']['label'][test_nonzero_idx])
    print(f"{X_test_emb.shape=} {y_test.shape=}")

    for model_name, model in models.items():
        print(f"Processing dataset: 'Mini' with embedding model: 'Sentence-BERT' "
            f"and classifier: '{model_name}'")
        model.fit(X_train_emb, y_train)
        y_pred = model.predict(X_test_emb)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
        accuracy = accuracy_score(y_test, y_pred)
        model_metrics = {
            "Preprocessing": "Mini",
            "Embedding Model": "Sentence-BERT",
            "Classifier": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1
        }
        for key in model_metrics:
            results[key].append(model_metrics[key])

evaluate_sbert(results, ds)

Sentence-BERT model loaded
X_train_emb.shape=(24922, 384) y_train.shape=(24922,)
X_test_emb.shape=(24923, 384) y_test.shape=(24923,)
Processing dataset: 'Mini' with embedding model: 'Sentence-BERT' and classifier: 'RandomForest'
Processing dataset: 'Mini' with embedding model: 'Sentence-BERT' and classifier: 'Logistic regression'
Processing dataset: 'Mini' with embedding model: 'Sentence-BERT' and classifier: 'XGBoost'
Processing dataset: 'Mini' with embedding model: 'Sentence-BERT' and classifier: 'SVC'


In [23]:
results_df = pd.DataFrame(results)

# Best results

### 1. Accuracy

In [24]:
results_df.sort_values(by="Accuracy", ascending=False).head(5)

,Preprocessing,Embedding Model,Classifier,Accuracy,Precision,Recall,F1-Score
19,Mini,Sentence-BERT,SVC,0.870601,0.873887,0.866244,0.870049
17,Mini,Sentence-BERT,Logistic regression,0.861453,0.865251,0.856295,0.860749
7,Lemmatized,Word2Vec,SVC,0.859280,0.860491,0.857600,0.859043
5,Lemmatized,Word2Vec,Logistic regression,0.844760,0.853035,0.833040,0.842919
18,Mini,Sentence-BERT,XGBoost,0.838302,0.843168,0.831261,0.837172


### Precision

In [25]:
results_df.sort_values(by="Precision", ascending=False).head(5)

,Preprocessing,Embedding Model,Classifier,Accuracy,Precision,Recall,F1-Score
19,Mini,Sentence-BERT,SVC,0.870601,0.873887,0.866244,0.870049
17,Mini,Sentence-BERT,Logistic regression,0.861453,0.865251,0.856295,0.860749
7,Lemmatized,Word2Vec,SVC,0.859280,0.860491,0.857600,0.859043
5,Lemmatized,Word2Vec,Logistic regression,0.844760,0.853035,0.833040,0.842919
18,Mini,Sentence-BERT,XGBoost,0.838302,0.843168,0.831261,0.837172


### Recall

In [26]:
results_df.sort_values(by="Recall", ascending=False).head(5)

,Preprocessing,Embedding Model,Classifier,Accuracy,Precision,Recall,F1-Score
19,Mini,Sentence-BERT,SVC,0.870601,0.873887,0.866244,0.870049
7,Lemmatized,Word2Vec,SVC,0.859280,0.860491,0.857600,0.859043
17,Mini,Sentence-BERT,Logistic regression,0.861453,0.865251,0.856295,0.860749
5,Lemmatized,Word2Vec,Logistic regression,0.844760,0.853035,0.833040,0.842919
18,Mini,Sentence-BERT,XGBoost,0.838302,0.843168,0.831261,0.837172


### F1-Score

In [27]:
results_df.sort_values(by="F1-Score", ascending=False).head(5)

,Preprocessing,Embedding Model,Classifier,Accuracy,Precision,Recall,F1-Score
19,Mini,Sentence-BERT,SVC,0.870601,0.873887,0.866244,0.870049
17,Mini,Sentence-BERT,Logistic regression,0.861453,0.865251,0.856295,0.860749
7,Lemmatized,Word2Vec,SVC,0.859280,0.860491,0.857600,0.859043
5,Lemmatized,Word2Vec,Logistic regression,0.844760,0.853035,0.833040,0.842919
18,Mini,Sentence-BERT,XGBoost,0.838302,0.843168,0.831261,0.837172


In [28]:
results_df.to_csv("task_2_results.csv", index=False)